# Solutions · 01 · Stencils and boundary conditions

Worked solutions to the exercises of [notebook 01](../notebooks/01_stencils_and_boundary_conditions.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere engmolines is missing): install it from GitHub
    import engmolines
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/engmolines"], check=True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from engmolines import grids, stencils as st, boundary as bc, operators as op, analysis
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

## Exercise 1

Build a fourth-order Laplacian with fourth-order one-sided closures and a fourth-order Laplacian with only second-order closures (use `fd_weights` on 4 points at the ends). Measure both orders. Which closure order does the global error follow, and why?

In [2]:
f = lambda x: np.sin(3 * x); d2 = lambda x: -9 * np.sin(3 * x)          # noqa: E731
def laplacian(x, closure_order):
    n = len(x); rows = {}
    for i in range(n):
        if 2 <= i <= n - 3:
            idx = np.arange(i - 2, i + 3)
        else:
            npts = 6 if closure_order == 4 else 4
            idx = np.arange(0, npts) if i < 2 else np.arange(n - npts, n)
        for j, wj in zip(idx, st.fd_weights(x[i], x[idx], 2)):
            rows.setdefault(j - i, np.zeros(n))[i] += wj
    offs = np.array(sorted(rows)); return st.StencilOperator(offs, np.array([rows[o] for o in offs]))
for closure in (4, 2):
    errs = [np.max(np.abs(laplacian(np.linspace(0, 1, n + 1), closure)(f(np.linspace(0, 1, n + 1))) - d2(np.linspace(0, 1, n + 1)))) for n in (20, 40, 80, 160)]
    print(f"closure order {closure}: errors {['%.1e' % e for e in errs]}, orders {np.round(analysis.observed_order(errs), 2)}")

closure order 4: errors ['1.5e-03', '6.3e-05', '3.0e-06', '1.5e-07'], orders [4.57 4.43 4.28]
closure order 2: errors ['5.6e-02', '1.0e-02', '2.1e-03', '4.7e-04'], orders [2.44 2.29 2.17]


With fourth-order closures the whole operator is fourth order; with second-order closures the global error
is second order - the boundary rows dominate the error, however good the interior stencil. (For time-dependent
problems the rule is slightly more forgiving: closures one order lower than the interior keep the global order
for parabolic problems, because the boundary error is smoothed; for the operator itself, as here, it is not.)

## Exercise 2

On a cell-centred grid impose u = 1 at x = 0 by the ghost formula and compare the resulting boundary flux k u_x with the exact one for the steady solution of u_xx = 0, u(0) = 1, u(1) = 0. What order is the flux error?

In [3]:
for n in (10, 20, 40, 80):
    g = grids.cell_centred(0, 1, n)
    u = 1 - g.x                                                     # the steady solution of u_xx = 0, u(0) = 1, u(1) = 0
    p = bc.pad(u, g.dx, bc.Dirichlet(1.0), bc.Dirichlet(0.0), kind="cell")
    flux = -(p[1] - p[0]) / g.dx                                    # -k u_x at the left face from the ghost
    print(f"n = {n:3d}: boundary flux {flux:.6f} (exact 1)")

n =  10: boundary flux 1.000000 (exact 1)
n =  20: boundary flux 1.000000 (exact 1)
n =  40: boundary flux 1.000000 (exact 1)
n =  80: boundary flux 1.000000 (exact 1)


For a linear solution the ghost formula is exact: the odd reflection about the face reproduces the linear
profile and its slope. For curved solutions the flux error is second order in Δx, like the ghost formula
itself.

## Exercise 3

**Implement it yourself:** write Fornberg's algorithm (or solve the Vandermonde system for the weights) for the first derivative on the points (0, 0.5, 1.2) at x = 0.3 and compare with `stencils.fd_weights`.

In [4]:
xs = np.array([0.0, 0.5, 1.2]); x0 = 0.3
V = np.vander(xs - x0, increasing=True).T                             # rows: powers of (x_j - x0)
rhs = np.array([0.0, 1.0, 0.0])                                      # we want the first derivative: d/dx of 1, (x - x0), (x - x0)^2 at x0
w = np.linalg.solve(V, rhs)
print("weights from the Vandermonde system:", w, "; Fornberg:", st.fd_weights(x0, xs, 1))

weights from the Vandermonde system: [-1.83333333  1.71428571  0.11904762] ; Fornberg: [-1.83333333  1.71428571  0.11904762]


Requiring the formula to be exact for 1, (x − x₀) and (x − x₀)² gives three linear equations for the three
weights; Fornberg's recurrence solves the same system stably for any number of points and any derivative.